# ❄️ Frozen Lake – A Gentle Introduction to Reinforcement Learning

<img src="img/frozen_lake.png" alt="Frozen Lake Problem" width="400"/>

Welcome to the Frozen Lake!

In this notebook we’ll use a tiny grid-world game to get an **intuitive** feel for some core ideas of **Reinforcement Learning (RL)**:

- An **agent** (our little elf) moves on a grid.
- The grid is the **environment** (the frozen lake).
- At each step, the agent chooses an **action** (left, down, right, up).
- The environment responds with a new **state** (where we are) and a **reward** (good or bad).
- Our goal is to understand how **good** it is to be in each state if we follow a certain **strategy**.

We’ll start by playing the game ourselves, then we’ll let simple strategies control the agent, and finally we’ll estimate how good each state is under a given strategy.

---

## 🗺️ Today’s Journey

Here’s the rough plan for this notebook:

| Section | What you’ll see | Key idea |
|--------|------------------|----------|
| **§1 – Meet the Environment** | You play Frozen Lake with the arrow keys | States, actions, rewards |
| **§2 – Strategies (Policies)** | Simple strategies that choose actions automatically | A strategy is a rule that picks actions |
| **§3 – Monte Carlo Rollouts** | Estimate “how good” each state is by averaging many episodes | Learn from complete episodes |
| **§4 – TD(0)** | Estimate “how good” each state is step by step | Learn from one step at a time (bootstrapping) |

> 💡 **Big picture:** We’re not trying to build the smartest possible agent.  
> We’re using Frozen Lake as a **playground** to see how strategies and “state values” behave in a small, visual world.


In [ ]:
import gymnasium as gym
import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

## 🧩 0. Registering the Frozen Lake Environment

In [ ]:
gym.register(
    id="FrozenLake-v2",
    entry_point="techdays26.frozen_lake.frozen_lake_enhanced:FrozenLakeEnv",
    kwargs={"map_name": "8x8"},
    max_episode_steps=200,
    reward_threshold=0.85,
)


def make_frozen_lake(render_mode="human", show_values=False, slippery=False):
    return gym.make(
        "FrozenLake-v2",
        desc=None,  # generate_random_map(size=8)
        map_name="8x8",
        show_q_labels=show_values,
        is_slippery=slippery,
        success_rate=3 / 4,
        reward_schedule=(1, -1, 0),
        render_mode=render_mode,
    )

## 🧊 1. Meet the Environment: Play Frozen Lake Yourself

Before we talk about strategies and value functions, let’s **experience** the environment.

- Use the **arrow keys** to move the elf.
- Try to reach the **goal** tile (G).
- Avoid the **holes** (H) – if you fall in, the episode ends.
- The reward is:
  - **+1** for reaching the goal
  - **−1** for falling into a hole
  - **0** for all other moves

In this first part, *you* are the strategy.  
Later, we’ll replace you with a function that chooses actions automatically.

In [ ]:
# https://github.com/johnnycode8/gym_solutions

import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

episodes: int = 2

env = make_frozen_lake()

try:
    env.unwrapped.set_info({
        "Mode": "Manual (Arrow Keys)",
    })

    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            # Wait for an arrow key press
            action = get_action_from_keyboard(key_to_action)
            if action is None:
                raise SystemExit  # on Escape

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## 🧭 2. Strategies (Policies): $\pi(a \mid s)$ in Plain Language

A **policy** (we’ll write it as $\pi$) is just a **strategy**:

> Given the current state, how likely are we to choose each possible action?

In code, we represent a policy as a function:

```python
def π(state, action) -> float:
    """Return the probability of taking `action` in `state`."""
```

To actually **pick** an action, we:

1. Ask the policy for the probability of each action.
2. Randomly choose one action according to these probabilities.

```python
def act(state, π):
    probs = [π(state, a) for a in range(4)]
    return np.random.choice([0, 1, 2, 3], p=probs)
```

So:

- `π` describes the **behavior** (“go down 45% of the time, right 45%, …”).
- `act` uses that behavior to pick a concrete action at each step.

### Example Strategies

Below we define two simple strategies:

**Question(s) for you:**  
- What do the following policies do? Can you guess?
- Before running the code, try to imagine what the elf will do with each strategy. Will it reach the goal? Will it get stuck?

In [ ]:
from techdays26.frozen_lake.frozen_lake_utils import action_to_arrow

print("Mapping of actions to directions:")
print(action_to_arrow)

In [ ]:
# To act, sample from the distribution:
def act(state, π):  # TODO: typing
    probs = [π(state, a) for a in range(4)]
    return np.random.choice([0, 1, 2, 3], p=probs)

In [ ]:
def π_1(state, action):
    """π(a|s) → probability"""
    #         ←    ↓    →    ↑
    probs = [0.05, 0.45, 0.45, 0.05]
    return probs[action]

In [ ]:
def π_2(state: int, action: int):
    """π(a|s) → probability"""
    if state % 8 < 7:
        #         ←    ↓    →    ↑
        probs = [0.0, 0.0, 1.0, 0.0]
    else:
        #         ←    ↓    →    ↑
        probs = [0.0, 1.0, 0.0, 0.0]

    return probs[action]

### 👀 2.1 Watching a Strategy in Action

In the next cell, you can choose:

- `π = π_1` or `π = π_2` for an **automatic** strategy, or
- `π = "manual"` to control the agent yourself with the arrow keys.

While it runs, pay attention to:

- The **path** the elf tends to take.
- How often it reaches the goal vs. falls into holes.
- How “smart” the behavior looks, given how simple the strategy is.

In [ ]:
episodes: int = 20
π = π_2  # Callable or "manual"

env = make_frozen_lake()

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            ####################################################################
            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)
            ####################################################################

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## 🎲 3. Estimating the Value Function $V^\pi(s)$ with Monte Carlo Rollouts

We now fix a strategy $\pi$ (for example, π₁ or π₂) and ask:

> If we start in a certain state and follow this strategy,  
> **how good is that state on average**?

“Good” here means:  
**average total reward we get from now until the episode ends**,  
with future rewards slightly discounted by a factor `gamma` (e.g. 0.9).

We call this number the **value** of a state under strategy $\pi$, written $V^\pi(s)$.  
You can think of it as: “How promising is this state if we keep following this strategy?”

### 💡 3.1 Monte Carlo Rollouts – Idea

We don’t try to compute this value with formulas.  
Instead, we **estimate** it by running the game many times:

1. Pick a strategy $\pi$ (manual or automatic).
2. Run an episode and record the states and rewards.
3. For each state, look at the **total reward from that point until the end** of the episode.
4. Repeat many episodes and **average** these totals for each state.

This is called:

- **Monte Carlo**: because we estimate averages by **random sampling** (like in a casino).
- **Rollouts**: because each episode is a “rollout” of the strategy in the environment.

In the next cell:

- You can use a **manual strategy** (`π = "manual"`) or a pre-defined one.
- We will update and display our current estimate of $V^\pi(s)$ on the map as we collect more episodes.

In [ ]:
episodes: int = 10

π = "manual"

env = make_frozen_lake(show_values=True)

################################################################################
nS = env.observation_space.n
v = np.zeros((nS,))
returns_sum = np.zeros(nS)
returns_count = np.zeros(nS)
gamma = 0.9  # discount factor γ
################################################################################

try:
    for i in range(episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False

        episode = []  # <--------------------------------------------------- NEW

        while not terminated and not truncated:
            env.unwrapped.set_v(v)  # <------------------------------------- NEW
            env.unwrapped.set_episode(i)

            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            ####################################################################
            # Append state and reward to this episode
            episode.append((state, reward))
            ####################################################################

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        ########################################################################
        # compute returns backwards and update first-visit states
        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s_t, r_t = episode[t]
            G = r_t + gamma * G
            if s_t not in visited:
                visited.add(s_t)
                returns_sum[s_t] += G
                returns_count[s_t] += 1
                v[s_t] = returns_sum[s_t] / returns_count[s_t]  # Expected value
        ########################################################################

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## 🔁 4. TD(0): Learning the Values $V^\pi(s)$ Step by Step

Monte Carlo looked at **whole episodes** and then updated the value of states.  
Now we’ll try a different approach that updates values **after every single step**.

The idea:

> The value of the current state should be close to  
> “reward we just got + value of the next state”.

So after each step, we adjust our value estimate for the current state a little bit towards:

```text
new estimate ≈ reward now + gamma * value of next state
```

This is called **TD(0)** (Temporal-Difference learning):

- **Temporal**: we learn from how things change over time.
- **Difference**: we look at the difference between what we expected and what actually happened.
- **0**: we only look one step ahead. We will later see how other values affect the learning

We control how fast we learn with:

- `alpha` (the **learning rate**): how big each update step is.
- `gamma` (the **discount factor**): how much we care about future rewards.

### 🧪 4.1 What We’ll Do Now

- Fix a strategy, e.g. `π = π_1`.
- Run many episodes.
- After each step, update our value estimate for the current state using TD(0).
- Show the evolving values $V^\pi(s)$ on the map.

You can compare this to the Monte Carlo version:

- **Monte Carlo**: waits until the end of the episode, then updates using the total reward.
- **TD(0)**: updates immediately after each step, using the next state’s current value estimate.

In [ ]:
episodes: int = 200000
π = "manual"

env = make_frozen_lake(show_values=True)

################################################################################
v = np.zeros((env.observation_space.n,))
gamma = 0.9  # discount factor γ
alpha = 0.01  # step size α
################################################################################

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_v(v)
            env.unwrapped.set_episode(i)

            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)
            new_state, reward, terminated, truncated, info_dict = env.step(action)

            # ---------- TD(0) policy evaluation update for V^π ----------
            done = terminated or truncated
            td_target = reward + (0.0 if done else gamma * v[new_state])
            td_error = td_target - v[state]
            v[state] += alpha * td_error
            # ------------------------------------------------------------

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## ✅ Summary

In this notebook you:

1. Played the **Frozen Lake** environment yourself.
2. Defined simple **strategies** (policies) that choose actions automatically.
3. Estimated how good each state is under a fixed strategy using:
   - **Monte Carlo rollouts** – learn from complete episodes by averaging total rewards.
   - **TD(0)** – learn step by step, using the next state’s current value estimate.

These ideas are the foundation for many RL algorithms such as **Q-learning** and **SARSA**.

If you have time, try:

- Changing the strategy $\pi$ and seeing how the values $V^\pi(s)$ change.
- Playing manually and watching how the value estimates adapt to your behavior.
- Tweaking `gamma` and `alpha` to see how they affect learning speed and stability.